# 企业RAG知识库 — 完整多模态文档处理

## 支持的文档类型

```
┌─────────────────────────────────────────────────────────────────┐
│                    输入文档（任意格式）                           │
│   PDF（数字版）  PDF（扫描版）  HTML/网页  .txt  .md           │
└───────────┬─────────────┬──────────────┬─────────────────────────┘
            │             │              │
            ▼             ▼              ▼
     ┌──────────┐  ┌──────────┐  ┌──────────────┐
     │ 文字层检测│  │PyMuPDF   │  │trafilatura   │
     │pdfplumber│  │渲染→图片 │  │+ BeautifulSoup│
     └──┬───┬───┘  └────┬─────┘  └──────┬───────┘
        │   │           │               │
        ▼   ▼           ▼               ▼
      文本  表格        OCR           Header-aware
        │   │       Tesseract         splitting
        │   ▼           │               │
        │ Markdown       │               │
        │ 表格格式        │               │
        │              清洗/去噪          │
        └──────┬─────────┴───────────────┘
               │
               ▼  图片另走视觉模型路径
     ┌─────────────────────┐
     │ RecursiveChar/HTML  │  按文档类型选分割符
     │ TextSplitter 分块   │
     └─────────┬───────────┘
               │
               ▼
     ┌─────────────────────┐
     │ HuggingFace         │  all-MiniLM-L6-v2
     │ Embeddings 向量化   │
     └─────────┬───────────┘
               │
               ▼
     ┌─────────────────────┐
     │ ChromaDB 持久化     │  text / table / image / html
     │ 分Collection存储    │  分Collection隔离
     └─────────┬───────────┘
               │
               ▼
     向量召回 Top-K → Cross-Encoder精排 → LLM生成
```

## Step 0：安装依赖

In [ ]:
%pip install -q \
    pdfplumber \
    pymupdf \
    pytesseract \
    Pillow \
    matplotlib \
    reportlab \
    trafilatura \
    beautifulsoup4 \
    lxml \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    chromadb \
    python-dotenv
print('依赖安装完成')
# Windows OCR额外要求：
#   Tesseract-OCR 引擎：https://github.com/UB-Mannheim/tesseract/wiki
#   中文语言包：安装时勾选 chi_sim（简体中文）

## Step 1：生成三类测试文件
- **数字PDF**：包含文本 + 表格 + 嵌入图表
- **扫描版PDF**：将数字PDF每页渲染为图片后重新打包（无文字层）
- **HTML网页**：含标题层级、正文、表格的企业文档

In [ ]:
import io, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import fitz  # PyMuPDF
from PIL import Image
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak, HRFlowable
)

DIGITAL_PDF = "digital_report.pdf"
SCANNED_PDF = "scanned_report.pdf"
HTML_FILE   = "knowledge_base.html"

styles = getSampleStyleSheet()
body   = styles["BodyText"]
h1, h2 = styles["Heading1"], styles["Heading2"]

# ── 生成柱状图 ────────────────────────────────────────────────
def make_bar_chart():
    fig, ax = plt.subplots(figsize=(6, 3))
    quarters = ["Q1", "Q2", "Q3", "Q4"]
    revenue  = [120, 145, 162, 198]
    cost     = [80, 90, 95, 110]
    x = np.arange(len(quarters))
    ax.bar(x - 0.2, revenue, 0.4, label="营收", color="#4C72B0")
    ax.bar(x + 0.2, cost,    0.4, label="成本", color="#DD8452")
    ax.set_xticks(x); ax.set_xticklabels(quarters)
    ax.set_title("2025年季度财务（百万元）"); ax.legend()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig); buf.seek(0)
    return buf

# ── 构建数字PDF ───────────────────────────────────────────────
story = []

# 页1：封面文本
story += [
    Paragraph("企业AI战略报告 2025", ParagraphStyle(
        "cover", parent=h1, fontSize=20, alignment=1, spaceAfter=20)),
    Paragraph(
        "本报告系统梳理了金融机构在生成式AI落地中的核心挑战与解决路径。"
        "RAG（检索增强生成）技术通过将企业私有知识库与大语言模型动态结合，"
        "有效解决了模型幻觉与知识时效性问题，成为银行、保险、券商的首选方案。"
        "三大核心场景（智能客服/合规审查/研究分析）预计18个月内实现ROI转正。", body),
    PageBreak(),
]

# 页2：业务效益表格
story += [
    Paragraph("第一章 业务场景效益对比", h1),
    Paragraph("下表汇总各AI应用场景实施前后核心指标变化：", body),
    Spacer(1, 0.4*cm),
]
tbl_data = [
    ["应用场景",     "实施前",       "实施后",       "提升",   "年化节省(万)"],
    ["智能客服",     "人工100%",     "人工62%",      "↑38%",  "1,200"],
    ["合规审查",     "自动化12%",    "自动化67%",    "↑55%",  "850"],
    ["研究报告",     "5个工作日",    "1.5个工作日",  "↓70%",  "320"],
    ["风控解释",     "人工100%",     "AI辅助80%",    "↑80%",  "480"],
]
t = Table(tbl_data, colWidths=[3.5*cm, 3.2*cm, 3.2*cm, 2*cm, 3.5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND",   (0,0), (-1,0), colors.HexColor("#2C5F8A")),
    ("TEXTCOLOR",    (0,0), (-1,0), colors.white),
    ("FONTNAME",     (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE",     (0,0), (-1,-1), 8),
    ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.HexColor("#EEF4FB"),colors.white]),
    ("GRID",         (0,0), (-1,-1), 0.5, colors.grey),
    ("ALIGN",        (0,0), (-1,-1), "CENTER"),
    ("TOPPADDING",   (0,0), (-1,-1), 5),
    ("BOTTOMPADDING",(0,0), (-1,-1), 5),
]))
story += [t, PageBreak()]

# 页3：图表
story += [
    Paragraph("第二章 财务数据", h1),
    Paragraph("图1展示2025年度各季度AI项目营收与成本对比，Q3起规模效应显现。", body),
    Spacer(1, 0.5*cm),
    RLImage(make_bar_chart(), width=14*cm, height=7*cm),
    Spacer(1, 0.3*cm),
    Paragraph("图1：2025年季度财务对比",
              ParagraphStyle("cap", parent=body, fontSize=9, alignment=1, textColor=colors.grey)),
    PageBreak(),
]

# 页4：技术选型表格
story += [
    Paragraph("第三章 向量数据库选型", h1),
]
tbl2 = [
    ["数据库",   "部署",       "扩展性", "混合检索", "推荐场景"],
    ["Chroma",   "本地/私有",  "中",     "否",       "POC/行内"],
    ["Weaviate", "云原生",     "高",     "是",       "生产环境"],
    ["Milvus",   "私有云",     "极高",   "是",       "大规模私有"],
    ["PGVector", "PostgreSQL", "中",     "否",       "已有PG"],
]
t2 = Table(tbl2, colWidths=[2.8*cm, 3*cm, 2.5*cm, 3*cm, 4*cm])
t2.setStyle(TableStyle([
    ("BACKGROUND",   (0,0), (-1,0), colors.HexColor("#1A6B3A")),
    ("TEXTCOLOR",    (0,0), (-1,0), colors.white),
    ("FONTSIZE",     (0,0), (-1,-1), 8),
    ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.HexColor("#EEF9F4"),colors.white]),
    ("GRID",         (0,0), (-1,-1), 0.5, colors.grey),
    ("ALIGN",        (0,0), (-1,-1), "CENTER"),
    ("TOPPADDING",   (0,0), (-1,-1), 5),
    ("BOTTOMPADDING",(0,0), (-1,-1), 5),
]))
story += [t2, PageBreak()]

# 页5：结论
story += [
    Paragraph("结论与建议", h1),
    Paragraph(
        "综合以上分析，建议企业AI知识库建设遵循以下四项原则。"
        "第一，数据主权优先，所有向量和文档存储于行内私有云，严禁发送至外部API。"
        "第二，模型选型差异化，中文场景优先BGE-M3，资源受限场景用MiniLM。"
        "第三，分场景建库，客服、合规、研究分析各建独立Collection，避免语义干扰。"
        "第四，持续运营，建立文档更新流水线，定期重新嵌入变更文档。", body),
]

doc = SimpleDocTemplate(DIGITAL_PDF, pagesize=A4,
                        leftMargin=2*cm, rightMargin=2*cm,
                        topMargin=2*cm, bottomMargin=2*cm)
doc.build(story)
print(f"数字PDF：{DIGITAL_PDF}  ({os.path.getsize(DIGITAL_PDF)/1024:.0f} KB, 5页)")

In [ ]:
# ── 生成扫描版PDF：将数字PDF每页渲染为图片，重新打包（无文字层）──
def create_scanned_pdf(src_path: str, dst_path: str, dpi: int = 150):
    """模拟扫描版PDF：渲染为图片再打包，Tesseract才能处理"""
    src = fitz.open(src_path)
    dst = fitz.open()  # 新建空白PDF
    scale = dpi / 72.0
    mat = fitz.Matrix(scale, scale)

    for page_num in range(len(src)):
        page   = src[page_num]
        pix    = page.get_pixmap(matrix=mat, alpha=False)
        # 页面点数 = 像素 / 缩放比例（保持A4物理尺寸）
        pw = page.rect.width
        ph = page.rect.height
        new_page = dst.new_page(width=pw, height=ph)
        new_page.insert_image(new_page.rect, stream=pix.tobytes("png"))

    dst.save(dst_path)
    src.close(); dst.close()

create_scanned_pdf(DIGITAL_PDF, SCANNED_PDF)
# 验证：扫描版应无文字层
import pdfplumber
with pdfplumber.open(SCANNED_PDF) as p:
    txt = p.pages[0].extract_text() or ""
print(f"扫描版PDF：{SCANNED_PDF}  ({os.path.getsize(SCANNED_PDF)/1024:.0f} KB)")
print(f"pdfplumber 文字层检测：{len(txt.strip())} 字符（0=纯图片，确认为扫描版）")

In [ ]:
# ── 生成测试HTML文件 ──────────────────────────────────────────
html_content = """<!DOCTYPE html>
<html lang="zh-CN">
<head><meta charset="UTF-8"><title>企业AI知识库管理规范 v2.0</title></head>
<body>

<nav id="navbar">导航栏 | 首页 | 关于 | 联系（此区域应被过滤）</nav>

<main>
  <h1>企业AI知识库管理规范</h1>
  <p>版本：v2.0 | 发布日期：2025-06-01 | 密级：内部</p>

  <h2>第一章 总则</h2>
  <p>为规范本行人工智能知识库的建设、维护与使用，保障知识资产的安全性和有效性，特制定本规范。</p>
  <p>本规范适用于全行各部门涉及AI知识库的相关工作，包括文档入库、版本管理、权限控制和检索使用。</p>

  <h2>第二章 文档分类与处理标准</h2>
  <h3>2.1 文档分类</h3>
  <p>根据文档来源和格式，分为以下四类，各类采用不同的处理策略：</p>

  <table border="1">
    <thead>
      <tr><th>文档类型</th><th>来源格式</th><th>处理工具</th><th>更新周期</th><th>优先级</th></tr>
    </thead>
    <tbody>
      <tr><td>产品说明书</td><td>PDF（数字版）</td><td>pdfplumber</td><td>季度</td><td>高</td></tr>
      <tr><td>监管公告</td><td>PDF（扫描版）</td><td>Tesseract OCR</td><td>实时</td><td>极高</td></tr>
      <tr><td>内部Wiki</td><td>HTML</td><td>trafilatura</td><td>实时</td><td>高</td></tr>
      <tr><td>研究报告</td><td>PDF（数字版）</td><td>pdfplumber+视觉</td><td>月度</td><td>中</td></tr>
      <tr><td>操作手册</td><td>Word/PDF</td><td>python-docx</td><td>年度</td><td>低</td></tr>
    </tbody>
  </table>

  <h3>2.2 质量标准</h3>
  <p>所有入库文档须满足：文字可读性不低于95%、表格结构完整、图片分辨率不低于150DPI。</p>

  <h2>第三章 安全与权限管理</h2>
  <h3>3.1 数据分级</h3>
  <p>知识库内容按照数据密级分为三个级别：公开、内部、机密。不同级别的内容存储在独立的Collection中，通过RBAC机制控制访问权限。</p>

  <h3>3.2 审计要求</h3>
  <p>所有检索操作均须记录操作日志，包含：查询时间、用户ID、查询内容（脱敏）、返回文档来源。日志保留期限不少于12个月。</p>

  <h2>第四章 向量数据库技术规范</h2>
  <p>本行统一采用ChromaDB作为向量数据库，部署于行内私有云，禁止使用外部SaaS服务。</p>

  <table border="1">
    <thead>
      <tr><th>Collection名称</th><th>内容类型</th><th>嵌入模型</th><th>chunk_size</th><th>overlap</th></tr>
    </thead>
    <tbody>
      <tr><td>cs_text</td><td>客服文本</td><td>BGE-M3</td><td>600</td><td>100</td></tr>
      <tr><td>cs_table</td><td>客服表格</td><td>BGE-M3</td><td>整表</td><td>0</td></tr>
      <tr><td>compliance_text</td><td>合规文本</td><td>BGE-M3</td><td>400</td><td>80</td></tr>
      <tr><td>research</td><td>研究报告</td><td>text-embedding-3</td><td>1000</td><td>200</td></tr>
    </tbody>
  </table>

  <h2>第五章 附则</h2>
  <p>本规范自发布之日起执行，由金融科技部负责解释和修订。如与其他规范冲突，以本规范为准。</p>
</main>

<footer>版权所有 © 2025 本行金融科技部（此区域应被过滤）</footer>

</body>
</html>"""

with open(HTML_FILE, "w", encoding="utf-8") as f:
    f.write(html_content)
print(f"HTML文件：{HTML_FILE}  ({os.path.getsize(HTML_FILE)/1024:.1f} KB)")
print("包含：5个标题层级 + 2个HTML表格 + 正文段落 + nav/footer噪音")

---
## 📌 核心参考：分割符配置手册

`RecursiveCharacterTextSplitter` 按**优先级从高到低**尝试分割符：  
- 前面的分割符语义粒度大（段落级），后面的粒度小（字符级）  
- 只有当分割后的块仍然超过 `chunk_size`，才继续用下一个分割符

In [ ]:
# ═══════════════════════════════════════════════════════════
#  分割符参考字典 — 按文档类型选择，复制到实际使用处
# ═══════════════════════════════════════════════════════════
SEPARATORS = {

    # ── 通用中文 / 中英混合（适合大多数企业文档）────────────
    "general_zh": [
        "\n\n",        # ① 段落空行（语义最完整，最优先）
        "\n# ",        # ② Markdown H1 标题前
        "\n## ",       # ③ Markdown H2 标题前
        "\n### ",      # ④ Markdown H3 标题前
        "。\n", "。",  # ⑤ 中文句号（行末优先，然后行中）
        "！\n", "！",  # ⑥ 中文叹号
        "？\n", "？",  # ⑦ 中文问号
        "；",           # ⑧ 中文分号（子句分隔）
        "\n",           # ⑨ 单换行（段内换行）
        ". ", "! ", "? ",  # ⑩ 英文句子
        "，",           # ⑪ 中文逗号（最后才用，语义较弱）
        " ",            # ⑫ 英文词间空格
        "",             # ⑬ 兜底：强制按字符数截断
    ],

    # ── 法律 / 合规条款（按条款编号分割）────────────────────
    "legal_zh": [
        "\n第",         # ① 条款编号（"第X条"、"第X章"）优先
        "\n（",         # ② 子条款 "（一）（二）"
        "\n\n",
        "。\n", "。",
        "；",
        "\n",
        " ", "",
    ],

    # ── 扫描版OCR输出（换行不规律，噪音多）────────────────────
    "ocr_output": [
        "\n\n",         # ① OCR段落间通常有多个换行，这是最可靠的边界
        "。",            # ② 中文句号是语义最可靠的切割点
        "！", "？",
        ". ",           # ③ 英文句子（OCR常识别英文比中文准）
        "\n",           # ④ 单换行（OCR错误断行，排后面）
        " ", "",
    ],

    # ── HTML正文（经过清洗后）─────────────────────────────────
    "html_text": [
        "\n\n",         # ① 块级元素换行
        "\n",
        "。", "！", "？",
        ". ", " ", "",
    ],

    # ── 纯英文技术文档 ────────────────────────────────────────
    "technical_en": [
        "\n\n",
        "\n## ", "\n### ",
        ". ", "! ", "? ",
        ";\n", "\n",
        " ", "",
    ],

    # ── 财务 / 数字密集型文档 ─────────────────────────────────
    "financial": [
        "\n\n",
        "。\n", "。",
        "；",
        "\n",
        ". ",
        " ", "",
        # ⚠️  故意不包含 "，" — 金融数字 1,234,567 含逗号，切割会破坏数字完整性
    ],
}

from langchain.text_splitter import RecursiveCharacterTextSplitter

def get_splitter(doc_type: str = "general_zh",
                 chunk_size: int = 800,
                 chunk_overlap: int = 150) -> RecursiveCharacterTextSplitter:
    """按文档类型返回配置好分割符的 splitter"""
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=SEPARATORS.get(doc_type, SEPARATORS["general_zh"]),
        length_function=len,
        add_start_index=True,
    )

# 打印分割符优先级示意
print("general_zh 分割符优先级（高→低）:")
for i, sep in enumerate(SEPARATORS["general_zh"], 1):
    display = repr(sep) if sep else "''（强制硬切）"
    print(f"  {i:2d}. {display}")

---
# Part 1 — 数字PDF（有文字层）

## ① 文本提取 + 分块

In [ ]:
import pdfplumber
from langchain.schema import Document

def extract_pdf_text(pdf_path: str, doc_type: str = "general_zh",
                     chunk_size: int = 800, overlap: int = 150) -> list:
    """
    从数字PDF提取文本并分块。
    pdfplumber相比PyPDFLoader优势：保留更多版式信息，中文断行处理更好。
    """
    splitter = get_splitter(doc_type, chunk_size, overlap)
    all_chunks = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            raw = page.extract_text()
            if not raw or len(raw.strip()) < 30:
                continue

            page_doc = Document(
                page_content=raw,
                metadata={
                    "source":       pdf_path,
                    "page":         page_num,          # 0-indexed
                    "content_type": "text",
                    "extract_method": "pdfplumber",
                    "page_width":   round(page.width, 1),
                    "page_height":  round(page.height, 1),
                }
            )
            chunks = splitter.split_documents([page_doc])
            all_chunks.extend(chunks)

    return all_chunks

text_docs = extract_pdf_text(DIGITAL_PDF, doc_type="general_zh")
print(f"文本chunks: {len(text_docs)} 个")
for i, c in enumerate(text_docs[:3]):
    print(f"\n[Chunk {i+1}] 第{c.metadata['page']+1}页 | 起始:{c.metadata.get('start_index',0)} | {len(c.page_content)}字")
    print(c.page_content[:200])

## ② 表格提取 → Markdown格式

**为什么转成 Markdown 而不是 CSV？**
- Markdown 保留表头语义，LLM 理解「第3行第2列」更准确
- CSV 的逗号分隔符在财务数字（1,234,567）中会产生误切
- 向量模型对结构化 Markdown 有更好的语义表示

In [ ]:
def table_to_markdown(table: list) -> str:
    """pdfplumber 二维列表 → Markdown 表格字符串"""
    if not table or len(table) < 2:
        return ""
    cleaned = [[str(c).strip() if c else "" for c in row] for row in table]
    n_cols  = max(len(r) for r in cleaned)
    widths  = [max(max(len(r[i]) if i < len(r) else 0 for r in cleaned), 3)
               for i in range(n_cols)]
    lines = []
    for idx, row in enumerate(cleaned):
        padded = row + [""] * (n_cols - len(row))
        cells  = [f" {c.ljust(widths[i])} " for i, c in enumerate(padded)]
        lines.append("|" + "|".join(cells) + "|")
        if idx == 0:  # 表头分隔线
            lines.append("|" + "|".join([":" + "-"*widths[i] + ":" for i in range(n_cols)]) + "|")
    return "\n".join(lines)


def extract_pdf_tables(pdf_path: str) -> list:
    """提取PDF中所有表格，转为Markdown，保留页码和位置元数据"""
    table_docs = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables     = page.extract_tables()
            tbl_finder = page.find_tables()  # 获取bbox
            if not tables:
                continue
            for tbl_idx, table in enumerate(tables):
                md = table_to_markdown(table)
                if len(md.strip()) < 20:
                    continue
                bbox = tbl_finder[tbl_idx].bbox if tbl_idx < len(tbl_finder) else None
                rows = len(table)
                cols = len(table[0]) if table else 0
                # 表格不做二次分块，整张表作为一个Document
                doc = Document(
                    page_content=md,
                    metadata={
                        "source":        pdf_path,
                        "page":          page_num,
                        "content_type":  "table",
                        "extract_method":"pdfplumber",
                        "table_index":   tbl_idx,
                        "rows":          rows,
                        "cols":          cols,
                        "bbox":          str(bbox),  # (x0,y0,x1,y1)
                    }
                )
                table_docs.append(doc)

    return table_docs


table_docs = extract_pdf_tables(DIGITAL_PDF)
print(f"表格Documents: {len(table_docs)} 个\n")
for d in table_docs:
    m = d.metadata
    print(f"第{m['page']+1}页 | 表格{m['table_index']+1} | {m['rows']}行×{m['cols']}列")
    print(d.page_content[:500])
    print()

## ③ 图片提取

### 路径 A — OCR（适合扫描件截图、含文字的图片）
### 路径 B — 视觉模型描述（适合图表、流程图、架构图）

In [ ]:
import re

IMG_DIR = "extracted_images"
os.makedirs(IMG_DIR, exist_ok=True)

def extract_pdf_images(pdf_path: str, min_size: int = 80) -> tuple:
    """
    用PyMuPDF提取PDF中嵌入的图片。
    返回: (image_paths列表, placeholder_docs列表)
    """
    pdf = fitz.open(pdf_path)
    paths, placeholder_docs = [], []

    for page_num in range(len(pdf)):
        page       = pdf[page_num]
        image_list = page.get_images(full=True)

        for img_idx, img_info in enumerate(image_list):
            xref     = img_info[0]
            base_img = pdf.extract_image(xref)
            w, h     = base_img["width"], base_img["height"]
            ext      = base_img["ext"]

            if w < min_size or h < min_size:
                continue  # 过滤装饰图标

            img_path = f"{IMG_DIR}/p{page_num+1}_img{img_idx+1}.{ext}"
            with open(img_path, "wb") as f:
                f.write(base_img["image"])
            paths.append(img_path)

            doc = Document(
                page_content=f"[图片] {pdf_path} 第{page_num+1}页 图{img_idx+1}",
                metadata={
                    "source":       pdf_path,
                    "page":         page_num,
                    "content_type": "image",
                    "image_path":   img_path,
                    "img_width":    w,
                    "img_height":   h,
                    "img_format":   ext,
                }
            )
            placeholder_docs.append(doc)
            print(f"  第{page_num+1}页 图{img_idx+1}: {w}×{h}px → {img_path}")

    pdf.close()
    return paths, placeholder_docs


image_paths, image_placeholder_docs = extract_pdf_images(DIGITAL_PDF)
print(f"\n共提取 {len(image_paths)} 张图片")

In [ ]:
# ── 路径A：Tesseract OCR ──────────────────────────────────────
# Windows 如未将Tesseract加入PATH，取消下行注释并修改路径：
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

import pytesseract

def ocr_image_file(img_path: str, lang: str = "chi_sim+eng") -> str:
    """
    对单张图片文件做OCR。
    lang参数：
      chi_sim        — 简体中文
      chi_tra        — 繁体中文
      chi_sim+eng    — 中英混合（推荐）
      eng            — 纯英文
    --psm 6: 假设页面是一整块均匀文本（适合全页OCR）
    --psm 3: 自动检测版面（适合多列布局）
    --oem 3: 使用LSTM引擎（精度最高）
    """
    try:
        img  = Image.open(img_path)
        text = pytesseract.image_to_string(img, lang=lang,
                                           config="--psm 6 --oem 3")
        return text.strip()
    except Exception as e:
        return f"OCR错误: {e}"


ocr_docs = []
for img_path, placeholder in zip(image_paths, image_placeholder_docs):
    ocr_text = ocr_image_file(img_path)
    if len(ocr_text) < 15 or ocr_text.startswith("OCR错误"):
        print(f"{img_path}: OCR结果为空（可能是纯图表，走视觉模型路径）")
        continue
    doc = Document(
        page_content=f"[图片OCR] {ocr_text}",
        metadata={**placeholder.metadata, "extract_method": "tesseract_ocr"}
    )
    ocr_docs.append(doc)
    print(f"{img_path}: OCR {len(ocr_text)} 字符 → {ocr_text[:100]}")

print(f"\nOCR成功: {len(ocr_docs)}/{len(image_paths)} 张")

In [ ]:
# ── 路径B：视觉模型描述图表 (Claude Vision) ───────────────────
import base64
from dotenv import load_dotenv
load_dotenv()

# 先展示图片，人工确认内容
from IPython.display import display, Image as IPImage
for p in image_paths:
    print(f"── {p} ──")
    display(IPImage(p, width=400))


def describe_image_claude(img_path: str) -> str:
    """
    调用Claude Vision描述图表内容。
    需要 .env 中设置 ANTHROPIC_API_KEY=sk-ant-xxx
    图表类型识别 + 数据解读 + 业务结论，适合柱状图/饼图/流程图。
    """
    import anthropic
    ext = img_path.rsplit(".", 1)[-1].lower()
    mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(ext, "image/png")
    with open(img_path, "rb") as f:
        b64 = base64.standard_b64encode(f.read()).decode()
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=600,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": mime, "data": b64}},
                {"type": "text",
                 "text": (
                     "请详细描述这张图表：\n"
                     "1. 图表类型（柱状图/折线图/饼图/流程图等）\n"
                     "2. 坐标轴或图例含义\n"
                     "3. 关键数据点和趋势\n"
                     "4. 主要业务结论\n"
                     "用中文回答，200字以内。"
                 )}
            ]
        }]
    )
    return msg.content[0].text


vision_docs = []
if os.getenv("ANTHROPIC_API_KEY"):
    for img_path, placeholder in zip(image_paths, image_placeholder_docs):
        desc = describe_image_claude(img_path)
        doc = Document(
            page_content=f"[图表描述] {desc}",
            metadata={**placeholder.metadata, "extract_method": "claude_vision"}
        )
        vision_docs.append(doc)
        print(f"{img_path}:\n  {desc[:200]}\n")
else:
    print("⚠️  未设置 ANTHROPIC_API_KEY，使用规则占位描述（生产环境请配置视觉模型）")
    for img_path, placeholder in zip(image_paths, image_placeholder_docs):
        m = placeholder.metadata
        desc = (f"来自《{m['source']}》第{m['page']+1}页的图表，"
                f"尺寸{m['img_width']}×{m['img_height']}像素。"
                f"建议配置视觉模型API以获取详细语义描述。")
        doc = Document(
            page_content=f"[图表描述] {desc}",
            metadata={**placeholder.metadata, "extract_method": "placeholder"}
        )
        vision_docs.append(doc)

print(f"图片视觉描述: {len(vision_docs)} 个Document")

---
# Part 2 — 扫描版PDF（无文字层）

## ① 自动检测是否为扫描版

In [ ]:
def is_scanned_pdf(pdf_path: str, sample_pages: int = 3, threshold: int = 50) -> bool:
    """
    检测PDF是否为扫描版（无可提取文字层）。
    策略：取前N页，若平均字符数 < threshold，判定为扫描版。
    threshold=50 意味着每页少于50个可识别字符即视为扫描版。
    """
    with pdfplumber.open(pdf_path) as pdf:
        pages_to_check = min(sample_pages, len(pdf.pages))
        total_chars = sum(
            len((pdf.pages[i].extract_text() or "").strip())
            for i in range(pages_to_check)
        )
    avg = total_chars / max(pages_to_check, 1)
    return avg < threshold


for path, label in [(DIGITAL_PDF, "数字PDF"), (SCANNED_PDF, "扫描版PDF")]:
    scanned = is_scanned_pdf(path)
    print(f"{label:10s} → is_scanned={scanned}  ({'需要OCR' if scanned else '有文字层，直接提取'})")

## ② 扫描版PDF处理：页面渲染 → OCR → 清洗 → 分块

In [ ]:
def clean_ocr_text(text: str) -> str:
    """
    OCR输出后处理：
    1. 去除孤立的噪音字符（竖线、斜线等OCR误识别）
    2. 修复被错误断开的中文行（中文行末不应有换行，除非是句末标点）
    3. 压缩多余空行
    """
    # 去除孤立噪音符号（连续2个以上且前后无中英文字符）
    text = re.sub(r"(?<![\w\u4e00-\u9fff])[|\\/_~`^]{2,}(?![\w\u4e00-\u9fff])", "", text)
    # 修复中文被错误断行：如果当前行末尾是中文字符（非句末标点），与下一行合并
    text = re.sub(r"([^\n。！？\.!?])\n([^\n])", r"\1\2", text)
    # 多余空行压缩为两个换行
    text = re.sub(r"\n{3,}", "\n\n", text)
    # 去除行首行尾空白（保留换行结构）
    lines = [line.strip() for line in text.split("\n")]
    return "\n".join(lines).strip()


def process_scanned_pdf(pdf_path: str,
                        lang: str = "chi_sim+eng",
                        dpi: int = 300,
                        psm: int = 6) -> list:
    """
    完整扫描版PDF处理流程：
      渲染（300dpi）→ Tesseract OCR → 文本清洗 → 分块

    dpi参数建议：
      150 — 速度优先，文字清晰的印刷体
      200 — 均衡（大多数情况）
      300 — 质量优先（手写体、低分辨率原件）
      400+ — 极低质量扫描件（速度很慢）

    psm（页面分割模式）：
      3  — 全自动版面检测（多列/混合布局）
      6  — 假设单块均匀文本（最常用）
      11 — 稀疏文本（表单、字段散布）
    """
    pdf      = fitz.open(pdf_path)
    splitter = get_splitter("ocr_output", chunk_size=600, overlap=100)
    all_docs = []
    scale    = dpi / 72.0
    mat      = fitz.Matrix(scale, scale)

    for page_num in range(len(pdf)):
        page = pdf[page_num]
        pix  = page.get_pixmap(matrix=mat, alpha=False)
        img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        try:
            raw = pytesseract.image_to_string(
                img, lang=lang,
                config=f"--psm {psm} --oem 3"
            )
        except Exception as e:
            print(f"第{page_num+1}页 OCR失败: {e}")
            continue

        cleaned = clean_ocr_text(raw)
        if len(cleaned) < 30:
            print(f"第{page_num+1}页: OCR结果过少（{len(cleaned)}字符），跳过")
            continue

        page_doc = Document(
            page_content=cleaned,
            metadata={
                "source":          pdf_path,
                "page":            page_num,
                "content_type":    "text",
                "extract_method": "tesseract_ocr",
                "ocr_lang":        lang,
                "ocr_dpi":         dpi,
                "ocr_psm":         psm,
            }
        )
        chunks = splitter.split_documents([page_doc])
        all_docs.extend(chunks)
        print(f"第{page_num+1}页: 渲染{pix.width}×{pix.height}px "
              f"→ OCR {len(cleaned)}字 → {len(chunks)} chunks")

    pdf.close()
    return all_docs


scanned_docs = process_scanned_pdf(SCANNED_PDF, dpi=150)  # 150dpi够用，省时间
print(f"\n扫描版PDF总计: {len(scanned_docs)} 个Document")

---
# Part 3 — HTML / 网页

## ① 文本提取（双策略：trafilatura 优先，BeautifulSoup 兜底）

In [ ]:
import trafilatura
from bs4 import BeautifulSoup

def extract_html_text(html_source: str, source_url: str = "") -> list:
    """
    从HTML提取正文文本（自动过滤导航栏/广告/页脚）。

    策略1 — trafilatura（首选）：
      专为新闻/博客/文档设计，自动识别主体内容区域。
      include_tables=False 因为表格单独处理。

    策略2 — BeautifulSoup（fallback）：
      手动移除噪音标签，提取剩余文本。
      适合trafilatura无法识别的内部wiki/企业系统页面。
    """
    # 尝试策略1
    extracted = trafilatura.extract(
        html_source,
        include_tables=False,
        include_images=False,
        include_links=False,
        no_fallback=False,
        favor_recall=True,   # 宁多勿少
    )
    if extracted and len(extracted.strip()) > 100:
        method = "trafilatura"
        clean_text = extracted
    else:
        # 策略2：BeautifulSoup 手动清洗
        soup = BeautifulSoup(html_source, "lxml")
        for tag in soup(["script", "style", "nav", "footer",
                         "header", "aside", "noscript", "iframe"]):
            tag.decompose()
        clean_text = soup.get_text(separator="\n", strip=True)
        method = "beautifulsoup"

    splitter = get_splitter("html_text", chunk_size=700, overlap=120)
    docs = splitter.create_documents(
        [clean_text],
        metadatas=[{
            "source":          source_url or "local_html",
            "page":            0,
            "content_type":    "text",
            "extract_method": f"html_{method}",
        }]
    )
    return docs


with open(HTML_FILE, encoding="utf-8") as f:
    html_raw = f.read()

html_text_docs = extract_html_text(html_raw, source_url=HTML_FILE)
print(f"HTML文本chunks: {len(html_text_docs)} 个")
for i, d in enumerate(html_text_docs):
    print(f"\n[{i+1}] 方法:{d.metadata['extract_method']} | {len(d.page_content)}字")
    print(d.page_content[:250])

## ② HTMLHeaderTextSplitter — 保留标题层级到元数据

In [ ]:
from langchain.text_splitter import HTMLHeaderTextSplitter

# 将HTML标题层级映射到元数据字段
headers_to_split = [
    ("h1", "section_h1"),
    ("h2", "section_h2"),
    ("h3", "section_h3"),
]
html_header_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split)

# 一次分割：先按标题切分，保留标题信息
header_splits = html_header_splitter.split_text(html_raw)

# 二次分割：对每个标题块再做文本分块（防止过长）
chunk_splitter = get_splitter("html_text", chunk_size=600, overlap=100)
html_header_docs = []
for split in header_splits:
    if len(split.page_content.strip()) < 20:
        continue
    sub_chunks = chunk_splitter.split_documents([split])
    for c in sub_chunks:
        c.metadata["source"]          = HTML_FILE
        c.metadata["content_type"]    = "text"
        c.metadata["extract_method"] = "html_header_splitter"
    html_header_docs.extend(sub_chunks)

print(f"HTMLHeaderTextSplitter 结果: {len(html_header_docs)} chunks（含标题层级元数据）\n")
for d in html_header_docs[:4]:
    section = d.metadata.get("section_h2") or d.metadata.get("section_h1") or "—"
    print(f"章节:{section:20s} | {len(d.page_content)}字 | {d.page_content[:120]}")

## ③ HTML表格提取 → Markdown

In [ ]:
def extract_html_tables(html_source: str, source_url: str = "") -> list:
    """提取HTML中所有<table>标签，转为Markdown格式Document"""
    soup = BeautifulSoup(html_source, "lxml")
    table_docs = []

    for tbl_idx, table in enumerate(soup.find_all("table")):
        rows = []
        for tr in table.find_all("tr"):
            cells = [td.get_text(separator=" ", strip=True)
                     for td in tr.find_all(["td", "th"])]
            if any(cells):  # 跳过空行
                rows.append(cells)

        if len(rows) < 2:
            continue

        md = table_to_markdown(rows)
        if not md.strip():
            continue

        # 尝试获取表格标题（前一个标题元素）
        caption = ""
        prev = table.find_previous(["h1","h2","h3","h4","caption"])
        if prev:
            caption = prev.get_text(strip=True)

        doc = Document(
            page_content=md,
            metadata={
                "source":          source_url or HTML_FILE,
                "page":            0,
                "content_type":    "table",
                "extract_method": "html_table",
                "table_index":     tbl_idx,
                "table_caption":   caption,
                "rows":            len(rows),
                "cols":            len(rows[0]),
            }
        )
        table_docs.append(doc)
        print(f"表格{tbl_idx+1}（{caption[:30]}）: {len(rows)}行×{len(rows[0])}列")
        print(md[:400])
        print()

    return table_docs


html_table_docs = extract_html_tables(html_raw, source_url=HTML_FILE)
print(f"HTML表格Documents: {len(html_table_docs)} 个")

---
# Part 4 — 统一处理器

封装成 `DocumentProcessor` 类，自动根据文件类型选择处理策略。

In [ ]:
class DocumentProcessor:
    """
    统一文档处理器。
    支持：PDF（数字版/扫描版）、HTML、TXT、MD。
    自动检测文档类型，选择最适合的处理策略。
    """

    SUPPORTED = {".pdf", ".html", ".htm", ".txt", ".md"}

    def process(self, file_path: str) -> dict:
        """
        处理单个文件，返回:
          {
            "text":  [Document, ...],   # 文本chunks
            "table": [Document, ...],   # 表格Documents
            "image": [Document, ...],   # 图片描述Documents
          }
        """
        ext = os.path.splitext(file_path)[1].lower()
        if ext not in self.SUPPORTED:
            raise ValueError(f"不支持的格式: {ext}，支持: {self.SUPPORTED}")

        print(f"\n处理文件: {file_path}")

        if ext == ".pdf":
            return self._process_pdf(file_path)
        elif ext in (".html", ".htm"):
            return self._process_html_file(file_path)
        elif ext in (".txt", ".md"):
            return self._process_text_file(file_path)

    def _process_pdf(self, path: str) -> dict:
        """PDF处理：自动区分数字版和扫描版"""
        if is_scanned_pdf(path):
            print("  → 检测为扫描版PDF，启用OCR")
            text_docs = process_scanned_pdf(path)
            return {"text": text_docs, "table": [], "image": []}
        else:
            print("  → 检测为数字PDF，直接提取")
            text_docs  = extract_pdf_text(path)
            table_docs = extract_pdf_tables(path)
            img_paths, img_placeholders = extract_pdf_images(path)
            # 图片：有API用视觉模型，否则用OCR兜底
            img_docs = []
            if os.getenv("ANTHROPIC_API_KEY"):
                for p, ph in zip(img_paths, img_placeholders):
                    d_text = describe_image_claude(p)
                    img_docs.append(Document(
                        page_content=f"[图表描述] {d_text}",
                        metadata={**ph.metadata, "extract_method": "claude_vision"}
                    ))
            else:
                for p, ph in zip(img_paths, img_placeholders):
                    ocr = ocr_image_file(p)
                    if len(ocr) > 15:
                        img_docs.append(Document(
                            page_content=f"[图片OCR] {ocr}",
                            metadata={**ph.metadata, "extract_method": "tesseract_ocr"}
                        ))
            print(f"  文本:{len(text_docs)} | 表格:{len(table_docs)} | 图片:{len(img_docs)}")
            return {"text": text_docs, "table": table_docs, "image": img_docs}

    def _process_html_file(self, path: str) -> dict:
        with open(path, encoding="utf-8") as f:
            html = f.read()
        text_docs  = extract_html_text(html, source_url=path)
        table_docs = extract_html_tables(html, source_url=path)
        print(f"  文本:{len(text_docs)} | 表格:{len(table_docs)}")
        return {"text": text_docs, "table": table_docs, "image": []}

    def _process_text_file(self, path: str) -> dict:
        with open(path, encoding="utf-8") as f:
            content = f.read()
        ext = os.path.splitext(path)[1].lower()
        dtype = "general_zh" if ext == ".txt" else "general_zh"
        splitter = get_splitter(dtype)
        docs = splitter.create_documents(
            [content],
            metadatas=[{"source": path, "page": 0, "content_type": "text",
                        "extract_method": "plain_text"}]
        )
        print(f"  文本:{len(docs)} chunks")
        return {"text": docs, "table": [], "image": []}


# ── 测试处理器 ────────────────────────────────────────────────
processor = DocumentProcessor()

results = {}
for file_path in [DIGITAL_PDF, SCANNED_PDF, HTML_FILE]:
    results[file_path] = processor.process(file_path)

print("\n" + "═"*55)
print("  DocumentProcessor 处理汇总")
print("═"*55)
total_text, total_table, total_image = 0, 0, 0
for path, res in results.items():
    t, tb, im = len(res["text"]), len(res["table"]), len(res["image"])
    total_text += t; total_table += tb; total_image += im
    print(f"  {os.path.basename(path):25s} 文本:{t:3d} 表格:{tb:2d} 图片:{im:2d}")
print("─"*55)
print(f"  总计{' '*23} 文本:{total_text:3d} 表格:{total_table:2d} 图片:{total_image:2d}")
print("═"*55)

---
# Part 5 — 向量化入库（ChromaDB）

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

CHROMA_DIR   = "./chroma_db"
COLLECTION_MAP = {
    "text":  "mm_complete_text",
    "table": "mm_complete_table",
    "image": "mm_complete_image",
}

print("加载嵌入模型...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("嵌入模型就绪 (all-MiniLM-L6-v2, 384维)\n")

# 汇总所有文档
all_by_type = {"text": [], "table": [], "image": []}
for res in results.values():
    for ctype in all_by_type:
        all_by_type[ctype].extend(res[ctype])

# 写入各Collection
stores = {}
for ctype, docs in all_by_type.items():
    if not docs:
        continue
    col_name = COLLECTION_MAP[ctype]
    stores[ctype] = Chroma.from_documents(
        docs, embeddings,
        collection_name=col_name,
        persist_directory=CHROMA_DIR
    )
    print(f"[{ctype:6s}] Collection={col_name:25s} 写入 {stores[ctype]._collection.count()} 条")

print("\n所有内容写入ChromaDB完成")

---
# Part 6 — 混合检索 + Cross-Encoder精排

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
ICON = {"text": "📄", "table": "📊", "image": "🖼️"}

def multimodal_search(query: str, k_per_store: int = 5, final_k: int = 4):
    """跨文本/表格/图片三个Collection检索，Cross-Encoder统一精排返回"""
    candidates = []
    for ctype, store in stores.items():
        docs = store.similarity_search(query, k=k_per_store)
        for d in docs:
            d.metadata["_ctype"] = ctype
        candidates.extend(docs)

    if not candidates:
        print("未找到相关内容")
        return []

    pairs  = [(query, d.page_content) for d in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)

    print(f"\n查询: {query!r}")
    print(f"召回: {len(candidates)} 条（跨{len(stores)}个Collection）→ 精排Top-{final_k}")
    print("─" * 70)
    for rank, (score, doc) in enumerate(ranked[:final_k], 1):
        ct    = doc.metadata.get("_ctype", "?")
        page  = doc.metadata.get("page", "?")
        src   = os.path.basename(doc.metadata.get("source", ""))
        meth  = doc.metadata.get("extract_method", "")
        print(f"[{rank}] {ICON.get(ct,'?')} {ct:6s} | 精排:{score:6.3f} | "
              f"{src} 第{page+1 if isinstance(page,int) else page}页 | {meth}")
        print(f"      {doc.page_content[:280]}")
        print()
    return [d for _, d in ranked[:final_k]]


# ── 测试查询 ─────────────────────────────────────────────────
multimodal_search("各业务场景AI实施前后效果对比数据")

In [ ]:
multimodal_search("知识库权限管理和安全审计要求")

In [ ]:
multimodal_search("2025年季度财务营收成本图表")

---
## 处理策略总结

| 文档类型 | 检测方式 | 提取工具 | 分割符配置 | chunk_size | 表格处理 | 图片处理 |
|----------|----------|----------|-----------|------------|----------|----------|
| 数字PDF | pdfplumber字符数>50 | pdfplumber | `general_zh` | 800 | →Markdown | OCR/视觉模型 |
| 扫描PDF | 字符数<50 | PyMuPDF渲染+Tesseract | `ocr_output` | 600 | 无（OCR） | 无（OCR） |
| HTML | 文件扩展名 | trafilatura+BS4 | `html_text` | 700 | →Markdown | 不处理 |
| TXT/MD | 文件扩展名 | 直接读取 | `general_zh` | 800 | 无 | 无 |

### 关键设计决策

1. **扫描版DPI选择**：150dpi速度快，文字清晰印刷品足够；手写体或模糊扫描件用300dpi
2. **OCR后处理必须做**：Tesseract中文常有断行错误，`clean_ocr_text()` 修复换行噪音
3. **表格不做二次分块**：整表作为单个Document，避免跨块查询时上下文丢失  
4. **图片双路径**：有视觉模型API走视觉描述（语义更好），无API自动降级到OCR
5. **分Collection存储**：text/table/image分开，查询时可按需选择或全量混合检索